In [3]:
# ============================================================
# CELL 1 — LOAD FEATURE SET V2
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/Users/akashray/Lithology_Predictor")

V2_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "engineered_features_v2.parquet"
)

V3_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "engineered_features_v3.parquet"
)


# ------------------------------------------------------------
# 2. Check that V2 exists
# ------------------------------------------------------------

if not V2_PATH.exists():
    raise FileNotFoundError(
        f"Feature Set V2 not found:\n{V2_PATH}"
    )


# ------------------------------------------------------------
# 3. Load V2
# ------------------------------------------------------------

df_v2 = pd.read_parquet(V2_PATH)


# ------------------------------------------------------------
# 4. Basic verification
# ------------------------------------------------------------

print("=" * 60)
print("FEATURE SET V2 — STARTING POINT")
print("=" * 60)

print(f"Path  : {V2_PATH}")
print(f"Shape : {df_v2.shape}")
print(f"Wells : {df_v2['WELL'].nunique():,}")
print(f"Rows  : {len(df_v2):,}")


# ------------------------------------------------------------
# 5. Verify required columns
# ------------------------------------------------------------

required_columns = [
    "WELL",
    "DEPT",
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB",
    "FORCE_2020_LITHOFACIES_LITHOLOGY",
]

missing = [
    c for c in required_columns
    if c not in df_v2.columns
]

if missing:
    raise ValueError(
        f"Required columns are missing: {missing}"
    )

print("\n✓ All required columns are present.")


# ------------------------------------------------------------
# 6. Check depth ordering within wells
# ------------------------------------------------------------

depth_ok = (
    df_v2
    .groupby("WELL")["DEPT"]
    .apply(lambda s: s.is_monotonic_increasing)
)

print(
    f"✓ Wells with valid increasing depth: "
    f"{depth_ok.sum()}/{len(depth_ok)}"
)

if not depth_ok.all():
    bad_wells = depth_ok[~depth_ok].index.tolist()

    raise ValueError(
        f"Depth ordering problem in wells: {bad_wells}"
    )


print("\nReady for finer rolling-window experiment.")

FEATURE SET V2 — STARTING POINT
Path  : /Users/akashray/Lithology_Predictor/data/processed/engineered_features_v2.parquet
Shape : (1430974, 52)
Wells : 118
Rows  : 1,430,974

✓ All required columns are present.
✓ Wells with valid increasing depth: 118/118

Ready for finer rolling-window experiment.


In [1]:
# ============================================================
# CELL 2 — DEFINE V3 ROLLING-WINDOW FEATURE PLAN
# ============================================================

# Existing W5 rolling windows
OLD_WINDOWS = [5, 9, 21]

# New finer windows
NEW_WINDOWS = [3, 7, 15]

# Complete V3 window set
V3_WINDOWS = [3, 5, 7, 9, 15, 21]

# Curves for rolling statistics
ROLLING_CURVES = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB",
]

# Statistics calculated for every curve/window combination
ROLLING_STATISTICS = [
    "mean",
    "std",
]

print("=" * 60)
print("V3 — FINER ROLLING-WINDOW EXPERIMENT")
print("=" * 60)

print("\nExisting windows:")
print(OLD_WINDOWS)

print("\nNew windows:")
print(NEW_WINDOWS)

print("\nFinal V3 windows:")
print(V3_WINDOWS)

print("\nRolling curves:")
print(ROLLING_CURVES)

print("\nRolling statistics:")
print(ROLLING_STATISTICS)


# ------------------------------------------------------------
# Calculate physical vertical scale
# ------------------------------------------------------------

DZ = 0.152

print("\nApproximate vertical scale:")

for window in V3_WINDOWS:
    print(
        f"{window:>2} samples "
        f"→ {window * DZ:.3f} m"
    )


# ------------------------------------------------------------
# Count the number of new features
# ------------------------------------------------------------

new_feature_count = (
    len(NEW_WINDOWS)
    * len(ROLLING_CURVES)
    * len(ROLLING_STATISTICS)
)

total_rolling_feature_count = (
    len(V3_WINDOWS)
    * len(ROLLING_CURVES)
    * len(ROLLING_STATISTICS)
)

print("\nNew rolling features to calculate:",
      new_feature_count)

print("Total rolling features in V3:",
      total_rolling_feature_count)

V3 — FINER ROLLING-WINDOW EXPERIMENT

Existing windows:
[5, 9, 21]

New windows:
[3, 7, 15]

Final V3 windows:
[3, 5, 7, 9, 15, 21]

Rolling curves:
['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB']

Rolling statistics:
['mean', 'std']

Approximate vertical scale:
 3 samples → 0.456 m
 5 samples → 0.760 m
 7 samples → 1.064 m
 9 samples → 1.368 m
15 samples → 2.280 m
21 samples → 3.192 m

New rolling features to calculate: 30
Total rolling features in V3: 60


In [4]:
# ============================================================
# CELL 3 — CALCULATE NEW V3 ROLLING FEATURES
# ============================================================

import time

# ------------------------------------------------------------
# Only calculate the NEW windows.
# Existing 5, 9, 21 features are already present in V2.
# ------------------------------------------------------------

new_rolling_features = pd.DataFrame(
    index=df_v2.index
)

print("=" * 60)
print("CALCULATING NEW V3 ROLLING FEATURES")
print("=" * 60)

print(f"Rows to process : {len(df_v2):,}")
print(f"Wells           : {df_v2['WELL'].nunique()}")
print(f"New windows     : {NEW_WINDOWS}")
print(f"Curves          : {ROLLING_CURVES}")

start_time = time.perf_counter()

# ------------------------------------------------------------
# Process one well at a time.
#
# This is essential because rolling geological context must
# never cross from one well into another.
# ------------------------------------------------------------

for well_idx, (well_name, well_df) in enumerate(
    df_v2.groupby("WELL", sort=False),
    start=1
):

    well_index = well_df.index

    # --------------------------------------------------------
    # Calculate rolling statistics for each curve
    # --------------------------------------------------------

    for curve in ROLLING_CURVES:

        series = well_df[curve]

        for window in NEW_WINDOWS:

            rolling = (
                series
                .rolling(
                    window=window,
                    center=True,
                    min_periods=3
                )
            )

            # Rolling mean
            new_rolling_features.loc[
                well_index,
                f"{curve}_ROLLMEAN_{window}"
            ] = rolling.mean().to_numpy()

            # Rolling standard deviation
            new_rolling_features.loc[
                well_index,
                f"{curve}_ROLLSTD_{window}"
            ] = rolling.std().to_numpy()

    # --------------------------------------------------------
    # Progress reporting
    # --------------------------------------------------------

    if (
        well_idx % 20 == 0
        or well_idx == df_v2["WELL"].nunique()
    ):
        print(
            f"Processed "
            f"{well_idx}/{df_v2['WELL'].nunique()} wells"
        )

elapsed = time.perf_counter() - start_time


# ------------------------------------------------------------
# Verify number of generated features
# ------------------------------------------------------------

expected_features = (
    len(ROLLING_CURVES)
    * len(NEW_WINDOWS)
    * len(ROLLING_STATISTICS)
)

print("\n" + "=" * 60)
print("NEW ROLLING FEATURE SUMMARY")
print("=" * 60)

print(
    f"Expected new features : {expected_features}"
)

print(
    f"Actual new features   : "
    f"{new_rolling_features.shape[1]}"
)

print(
    f"Calculation time      : "
    f"{elapsed:.2f} seconds"
)


# ------------------------------------------------------------
# Verify feature names
# ------------------------------------------------------------

print("\nNew feature names:")

for column in new_rolling_features.columns:
    print(" ", column)


# ------------------------------------------------------------
# Check row alignment
# ------------------------------------------------------------

if not new_rolling_features.index.equals(df_v2.index):
    raise ValueError(
        "Row-index alignment failed."
    )

print(
    "\n✓ New rolling features are aligned "
    "with V2 rows."
)

CALCULATING NEW V3 ROLLING FEATURES
Rows to process : 1,430,974
Wells           : 118
New windows     : [3, 7, 15]
Curves          : ['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB']
Processed 20/118 wells
Processed 40/118 wells
Processed 60/118 wells
Processed 80/118 wells
Processed 100/118 wells
Processed 118/118 wells

NEW ROLLING FEATURE SUMMARY
Expected new features : 30
Actual new features   : 30
Calculation time      : 5.06 seconds

New feature names:
  GR_ROLLMEAN_3
  GR_ROLLSTD_3
  GR_ROLLMEAN_7
  GR_ROLLSTD_7
  GR_ROLLMEAN_15
  GR_ROLLSTD_15
  RDEP_LOG10_ROLLMEAN_3
  RDEP_LOG10_ROLLSTD_3
  RDEP_LOG10_ROLLMEAN_7
  RDEP_LOG10_ROLLSTD_7
  RDEP_LOG10_ROLLMEAN_15
  RDEP_LOG10_ROLLSTD_15
  RMED_LOG10_ROLLMEAN_3
  RMED_LOG10_ROLLSTD_3
  RMED_LOG10_ROLLMEAN_7
  RMED_LOG10_ROLLSTD_7
  RMED_LOG10_ROLLMEAN_15
  RMED_LOG10_ROLLSTD_15
  DTC_ROLLMEAN_3
  DTC_ROLLSTD_3
  DTC_ROLLMEAN_7
  DTC_ROLLSTD_7
  DTC_ROLLMEAN_15
  DTC_ROLLSTD_15
  RHOB_ROLLMEAN_3
  RHOB_ROLLSTD_3
  RHOB_ROLLMEAN_7
  

In [5]:
# ============================================================
# CELL 4 — BUILD FEATURE SET V3
# ============================================================

print("=" * 60)
print("BUILDING FEATURE SET V3")
print("=" * 60)


# ------------------------------------------------------------
# 1. Make a copy of V2
# ------------------------------------------------------------

df_v3 = df_v2.copy()


# ------------------------------------------------------------
# 2. Prevent accidental duplicate columns
# ------------------------------------------------------------

duplicate_columns = [
    col
    for col in new_rolling_features.columns
    if col in df_v3.columns
]

if duplicate_columns:
    raise ValueError(
        "These new feature columns already exist in V2:\n"
        f"{duplicate_columns}"
    )


# ------------------------------------------------------------
# 3. Add the 30 new rolling features
# ------------------------------------------------------------

df_v3 = pd.concat(
    [
        df_v3,
        new_rolling_features
    ],
    axis=1
)


# ------------------------------------------------------------
# 4. Verify row count
# ------------------------------------------------------------

if len(df_v3) != len(df_v2):
    raise ValueError(
        "Row count changed during V3 construction."
    )


# ------------------------------------------------------------
# 5. Verify WELL alignment
# ------------------------------------------------------------

if not df_v3["WELL"].equals(df_v2["WELL"]):
    raise ValueError(
        "WELL alignment changed during V3 construction."
    )


# ------------------------------------------------------------
# 6. Verify target alignment
# ------------------------------------------------------------

target_col = "FORCE_2020_LITHOFACIES_LITHOLOGY"

if not df_v3[target_col].equals(
    df_v2[target_col]
):
    raise ValueError(
        "Lithology target alignment changed."
    )


# ------------------------------------------------------------
# 7. Count feature columns
# ------------------------------------------------------------

print(f"V2 shape : {df_v2.shape}")
print(f"V3 shape : {df_v3.shape}")

print(
    f"\nAdded columns : "
    f"{df_v3.shape[1] - df_v2.shape[1]}"
)


# ------------------------------------------------------------
# 8. Check expected feature count
# ------------------------------------------------------------

expected_v3_columns = (
    df_v2.shape[1] + 30
)

if df_v3.shape[1] != expected_v3_columns:
    raise ValueError(
        f"Unexpected V3 column count. "
        f"Expected {expected_v3_columns}, "
        f"got {df_v3.shape[1]}."
    )


# ------------------------------------------------------------
# 9. Verify new feature missingness
# ------------------------------------------------------------

print("\nNew rolling feature missingness (%):")

v3_new_missing = (
    df_v3[new_rolling_features.columns]
    .isna()
    .mean()
    .mul(100)
    .round(3)
)

print(v3_new_missing)


# ------------------------------------------------------------
# 10. Verify infinite values
# ------------------------------------------------------------

numeric_v3 = df_v3.select_dtypes(
    include=np.number
)

infinite_count = np.isinf(
    numeric_v3.to_numpy()
).sum()

print(
    f"\nInfinite numeric values: "
    f"{infinite_count}"
)

if infinite_count != 0:
    raise ValueError(
        "Infinite values found in V3."
    )


# ------------------------------------------------------------
# 11. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("V3 BUILD VERIFICATION")
print("=" * 60)

print(f"Rows  : {len(df_v3):,}")
print(f"Wells : {df_v3['WELL'].nunique()}")
print(f"Cols  : {df_v3.shape[1]}")

print("\n✓ Row count preserved.")
print("✓ WELL alignment preserved.")
print("✓ Target alignment preserved.")
print("✓ 30 new rolling features added.")
print("✓ No infinite numeric values.")

BUILDING FEATURE SET V3
V2 shape : (1430974, 52)
V3 shape : (1430974, 82)

Added columns : 30

New rolling feature missingness (%):
GR_ROLLMEAN_3              0.107
GR_ROLLSTD_3               0.107
GR_ROLLMEAN_7              0.087
GR_ROLLSTD_7               0.087
GR_ROLLMEAN_15             0.080
GR_ROLLSTD_15              0.080
RDEP_LOG10_ROLLMEAN_3      0.793
RDEP_LOG10_ROLLSTD_3       0.793
RDEP_LOG10_ROLLMEAN_7      0.772
RDEP_LOG10_ROLLSTD_7       0.772
RDEP_LOG10_ROLLMEAN_15     0.764
RDEP_LOG10_ROLLSTD_15      0.764
RMED_LOG10_ROLLMEAN_3      3.501
RMED_LOG10_ROLLSTD_3       3.501
RMED_LOG10_ROLLMEAN_7      3.417
RMED_LOG10_ROLLSTD_7       3.417
RMED_LOG10_ROLLMEAN_15     3.348
RMED_LOG10_ROLLSTD_15      3.348
DTC_ROLLMEAN_3             6.068
DTC_ROLLSTD_3              6.068
DTC_ROLLMEAN_7             6.040
DTC_ROLLSTD_7              6.040
DTC_ROLLMEAN_15            6.004
DTC_ROLLSTD_15             6.004
RHOB_ROLLMEAN_3           13.164
RHOB_ROLLSTD_3            13.164
RHOB_ROLLM

In [6]:
# ============================================================
# CELL 5 — V3 ROLLING-SCALE INFORMATION CHECK
# ============================================================

print("=" * 60)
print("V3 — ROLLING-SCALE INFORMATION CHECK")
print("=" * 60)


# ------------------------------------------------------------
# 1. Select rolling MEAN features
# ------------------------------------------------------------

mean_features = []

for curve in ROLLING_CURVES:
    for window in V3_WINDOWS:
        mean_features.append(
            f"{curve}_ROLLMEAN_{window}"
        )


# ------------------------------------------------------------
# 2. Select rolling STD features
# ------------------------------------------------------------

std_features = []

for curve in ROLLING_CURVES:
    for window in V3_WINDOWS:
        std_features.append(
            f"{curve}_ROLLSTD_{window}"
        )


# ------------------------------------------------------------
# 3. Calculate correlation matrices
# ------------------------------------------------------------

mean_corr = df_v3[
    mean_features
].corr()

std_corr = df_v3[
    std_features
].corr()


# ------------------------------------------------------------
# 4. Show correlations for each curve separately
# ------------------------------------------------------------

for curve in ROLLING_CURVES:

    curve_mean_features = [
        f"{curve}_ROLLMEAN_{w}"
        for w in V3_WINDOWS
    ]

    curve_std_features = [
        f"{curve}_ROLLSTD_{w}"
        for w in V3_WINDOWS
    ]

    print("\n" + "=" * 60)
    print(f"{curve} — ROLLING MEAN CORRELATION")
    print("=" * 60)

    display(
        mean_corr.loc[
            curve_mean_features,
            curve_mean_features
        ].round(3)
    )

    print(f"\n{curve} — ROLLING STD CORRELATION")

    display(
        std_corr.loc[
            curve_std_features,
            curve_std_features
        ].round(3)
    )


# ------------------------------------------------------------
# 5. Compare adjacent window differences
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MEAN ABSOLUTE DIFFERENCE BETWEEN ROLLING MEANS")
print("=" * 60)

for curve in ROLLING_CURVES:

    print(f"\n{curve}")

    for w1, w2 in zip(
        V3_WINDOWS[:-1],
        V3_WINDOWS[1:]
    ):

        f1 = f"{curve}_ROLLMEAN_{w1}"
        f2 = f"{curve}_ROLLMEAN_{w2}"

        valid = df_v3[
            [f1, f2]
        ].dropna()

        mean_difference = (
            valid[f1] - valid[f2]
        ).abs().mean()

        print(
            f"  {w1:>2} vs {w2:<2} "
            f"→ mean absolute difference = "
            f"{mean_difference:.6f}"
        )

V3 — ROLLING-SCALE INFORMATION CHECK

GR — ROLLING MEAN CORRELATION


,GR_ROLLMEAN_3,GR_ROLLMEAN_5,GR_ROLLMEAN_7,GR_ROLLMEAN_9,GR_ROLLMEAN_15,GR_ROLLMEAN_21
GR_ROLLMEAN_3,1.000,0.999,0.998,0.995,0.989,0.984
GR_ROLLMEAN_5,0.999,1.000,0.999,0.998,0.992,0.988
GR_ROLLMEAN_7,0.998,0.999,1.000,1.000,0.995,0.991
GR_ROLLMEAN_9,0.995,0.998,1.000,1.000,0.997,0.993
GR_ROLLMEAN_15,0.989,0.992,0.995,0.997,1.000,0.998
GR_ROLLMEAN_21,0.984,0.988,0.991,0.993,0.998,1.000



GR — ROLLING STD CORRELATION


,GR_ROLLSTD_3,GR_ROLLSTD_5,GR_ROLLSTD_7,GR_ROLLSTD_9,GR_ROLLSTD_15,GR_ROLLSTD_21
GR_ROLLSTD_3,1.000,0.949,0.876,0.807,0.675,0.603
GR_ROLLSTD_5,0.949,1.000,0.969,0.914,0.774,0.692
GR_ROLLSTD_7,0.876,0.969,1.000,0.979,0.853,0.764
GR_ROLLSTD_9,0.807,0.914,0.979,1.000,0.915,0.824
GR_ROLLSTD_15,0.675,0.774,0.853,0.915,1.000,0.952
GR_ROLLSTD_21,0.603,0.692,0.764,0.824,0.952,1.000



RDEP_LOG10 — ROLLING MEAN CORRELATION


...


RHOB — ROLLING MEAN CORRELATION


,RHOB_ROLLMEAN_3,RHOB_ROLLMEAN_5,RHOB_ROLLMEAN_7,RHOB_ROLLMEAN_9,RHOB_ROLLMEAN_15,RHOB_ROLLMEAN_21
RHOB_ROLLMEAN_3,1.000,0.999,0.997,0.994,0.986,0.981
RHOB_ROLLMEAN_5,0.999,1.000,0.999,0.997,0.991,0.985
RHOB_ROLLMEAN_7,0.997,0.999,1.000,0.999,0.994,0.989
RHOB_ROLLMEAN_9,0.994,0.997,0.999,1.000,0.997,0.992
RHOB_ROLLMEAN_15,0.986,0.991,0.994,0.997,1.000,0.998
RHOB_ROLLMEAN_21,0.981,0.985,0.989,0.992,0.998,1.000



RHOB — ROLLING STD CORRELATION


,RHOB_ROLLSTD_3,RHOB_ROLLSTD_5,RHOB_ROLLSTD_7,RHOB_ROLLSTD_9,RHOB_ROLLSTD_15,RHOB_ROLLSTD_21
RHOB_ROLLSTD_3,1.000,0.958,0.887,0.816,0.676,0.600
RHOB_ROLLSTD_5,0.958,1.000,0.971,0.914,0.766,0.681
RHOB_ROLLSTD_7,0.887,0.971,1.000,0.978,0.844,0.752
RHOB_ROLLSTD_9,0.816,0.914,0.978,1.000,0.909,0.813
RHOB_ROLLSTD_15,0.676,0.766,0.844,0.909,1.000,0.947
RHOB_ROLLSTD_21,0.600,0.681,0.752,0.813,0.947,1.000



MEAN ABSOLUTE DIFFERENCE BETWEEN ROLLING MEANS

GR
   3 vs 5  → mean absolute difference = 0.768594
   5 vs 7  → mean absolute difference = 0.703391
   7 vs 9  → mean absolute difference = 0.618285
   9 vs 15 → mean absolute difference = 1.367283
  15 vs 21 → mean absolute difference = 1.042054

RDEP_LOG10
   3 vs 5  → mean absolute difference = 0.004442
   5 vs 7  → mean absolute difference = 0.004592
   7 vs 9  → mean absolute difference = 0.004466
   9 vs 15 → mean absolute difference = 0.012101
  15 vs 21 → mean absolute difference = 0.010553

RMED_LOG10
   3 vs 5  → mean absolute difference = 0.006519
   5 vs 7  → mean absolute difference = 0.006508
   7 vs 9  → mean absolute difference = 0.006037
   9 vs 15 → mean absolute difference = 0.013970
  15 vs 21 → mean absolute difference = 0.010969

DTC
   3 vs 5  → mean absolute difference = 0.391421
   5 vs 7  → mean absolute difference = 0.405504
   7 vs 9  → mean absolute difference = 0.379006
   9 vs 15 → mean absolute difference

In [8]:
# ============================================================
# V3 — DIAGNOSE EXTRA COLUMNS
# ============================================================

print("=" * 70)
print("V3 — COLUMN AUDIT")
print("=" * 70)

print(f"\nTotal columns in df_v3: {len(df_v3.columns)}")

print("\nALL COLUMNS:")
for i, col in enumerate(df_v3.columns, start=1):
    print(f"{i:3d}. {col}")

# ------------------------------------------------------------
# Columns currently excluded by our metadata definition
# ------------------------------------------------------------

META_COLS = {"WELL", "DEPT", "target", "SPLIT"}

candidate_features = [
    col for col in df_v3.columns
    if col not in META_COLS
]

print("\n" + "=" * 70)
print(f"Columns after excluding {META_COLS}: {len(candidate_features)}")
print("=" * 70)

# ------------------------------------------------------------
# Show likely metadata / non-feature columns
# ------------------------------------------------------------

print("\nPotential extra columns:")
for col in candidate_features:
    print(f"  {col}")

print("\n" + "=" * 70)
print("Expected:")
print("  Original V2 ML features : 46")
print("  New rolling features    : 30")
print("  True V3 ML features     : 76")
print("  Currently detected      :", len(candidate_features))
print("=" * 70)

V3 — COLUMN AUDIT

Total columns in df_v3: 82

ALL COLUMNS:
  1. GR
  2. RDEP
  3. RMED
  4. DTC
  5. RHOB
  6. FORCE_2020_LITHOFACIES_LITHOLOGY
  7. WELL
  8. RDEP_LOG10
  9. RMED_LOG10
 10. GR_MISSING
 11. RDEP_LOG10_MISSING
 12. RMED_LOG10_MISSING
 13. DTC_MISSING
 14. RHOB_MISSING
 15. SPLIT
 16. GR_ROLLMEAN_5
 17. GR_ROLLSTD_5
 18. GR_ROLLMEAN_9
 19. GR_ROLLSTD_9
 20. GR_ROLLMEAN_21
 21. GR_ROLLSTD_21
 22. RDEP_LOG10_ROLLMEAN_5
 23. RDEP_LOG10_ROLLSTD_5
 24. RDEP_LOG10_ROLLMEAN_9
 25. RDEP_LOG10_ROLLSTD_9
 26. RDEP_LOG10_ROLLMEAN_21
 27. RDEP_LOG10_ROLLSTD_21
 28. RMED_LOG10_ROLLMEAN_5
 29. RMED_LOG10_ROLLSTD_5
 30. RMED_LOG10_ROLLMEAN_9
 31. RMED_LOG10_ROLLSTD_9
 32. RMED_LOG10_ROLLMEAN_21
 33. RMED_LOG10_ROLLSTD_21
 34. DTC_ROLLMEAN_5
 35. DTC_ROLLSTD_5
 36. DTC_ROLLMEAN_9
 37. DTC_ROLLSTD_9
 38. DTC_ROLLMEAN_21
 39. DTC_ROLLSTD_21
 40. RHOB_ROLLMEAN_5
 41. RHOB_ROLLSTD_5
 42. RHOB_ROLLMEAN_9
 43. RHOB_ROLLSTD_9
 44. RHOB_ROLLMEAN_21
 45. RHOB_ROLLSTD_21
 46. RESISTIVITY_SEPARAT

In [10]:
# ============================================================
# V3 — DIAGNOSE DEVELOPMENT-SPLIT WELL NAME MISMATCH
# ============================================================

print("=" * 70)
print("V3 — DEVELOPMENT SPLIT WELL-NAME AUDIT")
print("=" * 70)

# Wells in V3
v3_wells = set(df_v3["WELL"].dropna().unique())

# Wells in development split
split_wells = set(dev_split["WELL"].dropna().unique())

print(f"\nUnique wells in df_v3:              {len(v3_wells)}")
print(f"Unique wells in development_split:  {len(split_wells)}")

# ------------------------------------------------------------
# Wells present in V3 but absent from development split
# ------------------------------------------------------------

missing_from_split = sorted(v3_wells - split_wells)

print(f"\nWells in V3 but NOT in development_split: {len(missing_from_split)}")

for well in missing_from_split:
    print(f"  {repr(well)}")

# ------------------------------------------------------------
# Wells present in development split but absent from V3
# ------------------------------------------------------------

missing_from_v3 = sorted(split_wells - v3_wells)

print(f"\nWells in development_split but NOT in V3: {len(missing_from_v3)}")

for well in missing_from_v3:
    print(f"  {repr(well)}")

# ------------------------------------------------------------
# Show examples from both sides
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 20 V3 WELL NAMES")
print("=" * 70)

for well in sorted(v3_wells)[:20]:
    print(repr(well))

print("\n" + "=" * 70)
print("FIRST 20 SPLIT WELL NAMES")
print("=" * 70)

for well in sorted(split_wells)[:20]:
    print(repr(well))

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)

V3 — DEVELOPMENT SPLIT WELL-NAME AUDIT

Unique wells in df_v3:              118
Unique wells in development_split:  98

Wells in V3 but NOT in development_split: 20
  '15_9-14'
  '15_9-23'
  '16_2-7'
  '16_7-6'
  '17_4-1'
  '25_10-10'
  '25_10-9'
  '25_11-24'
  '25_5-3'
  '29_3-1'
  '31_2-10'
  '31_2-21 S'
  '34_10-16 R'
  '34_3-2 S'
  '34_3-3 A'
  '34_6-1 S'
  '35_11-5'
  '35_6-2 S'
  '35_9-7'
  '35_9-8'

Wells in development_split but NOT in V3: 0

FIRST 20 V3 WELL NAMES
'15_9-13'
'15_9-14'
'15_9-15'
'15_9-17'
'15_9-23'
'16_1-2'
'16_1-6 A'
'16_10-1'
'16_10-2'
'16_10-3'
'16_10-5'
'16_11-1 ST3'
'16_2-11 A'
'16_2-16'
'16_2-6'
'16_2-7'
'16_4-1'
'16_5-3'
'16_7-4'
'16_7-5'

FIRST 20 SPLIT WELL NAMES
'15_9-13'
'15_9-15'
'15_9-17'
'16_1-2'
'16_1-6 A'
'16_10-1'
'16_10-2'
'16_10-3'
'16_10-5'
'16_11-1 ST3'
'16_2-11 A'
'16_2-16'
'16_2-6'
'16_4-1'
'16_5-3'
'16_7-4'
'16_7-5'
'16_8-1'
'17_11-1'
'25_11-15'

AUDIT COMPLETE


In [12]:
# ============================================================
# OFFICIAL FORCE-2020 SPLIT — AUDIT ACTUAL VALUES
# ============================================================

print("=" * 70)
print("OFFICIAL SPLIT AUDIT")
print("=" * 70)

official_split = pd.read_csv(
    "../data/splits/force2020_split.csv"
)

print("\nColumns:")
print(official_split.columns.tolist())

print("\nUnique SPLIT values:")
print(
    official_split["SPLIT"]
    .value_counts(dropna=False)
)

print("\nExact unique labels:")
for value in official_split["SPLIT"].drop_duplicates():
    print(repr(value))

print("\nTotal rows:", len(official_split))
print("Unique wells:", official_split["WELL"].nunique())

print("\nFirst 20 rows:")
display(official_split.head(20))

print("\n" + "=" * 70)
print("END AUDIT")
print("=" * 70)

OFFICIAL SPLIT AUDIT

Columns:
['WELL', 'SPLIT']

Unique SPLIT values:
SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64

Exact unique labels:
'TRAIN'
'PUBLIC'
'BLIND_TEST'

Total rows: 118
Unique wells: 118

First 20 rows:


,WELL,SPLIT
0,15_9-13,TRAIN
1,15_9-14,PUBLIC
2,15_9-15,TRAIN
3,15_9-17,TRAIN
4,15_9-23,BLIND_TEST
5,16_1-2,TRAIN
6,16_1-6 A,TRAIN
7,16_10-1,TRAIN
8,16_10-2,TRAIN
9,16_10-3,TRAIN



END AUDIT


In [15]:
# ============================================================
# DIAGNOSE CURRENT NOTEBOOK STATE
# ============================================================

print("=" * 70)
print("CURRENT DATAFRAME STATE")
print("=" * 70)

print("\ndf_v3_ml columns containing 'SPLIT':")
print([
    col for col in df_v3_ml.columns
    if "SPLIT" in col.upper()
])

print("\ndf_v3_train_pool columns containing 'SPLIT':")
if "df_v3_train_pool" in globals():
    print([
        col for col in df_v3_train_pool.columns
        if "SPLIT" in col.upper()
    ])
else:
    print("df_v3_train_pool does not exist")

print("\nDevelopment split columns:")
print(dev_split.columns.tolist())

print("\nDevelopment split shape:")
print(dev_split.shape)

print("\nDevelopment split counts:")
print(dev_split["DEV_SPLIT"].value_counts())

print("=" * 70)

CURRENT DATAFRAME STATE

df_v3_ml columns containing 'SPLIT':
['DEV_SPLIT']

df_v3_train_pool columns containing 'SPLIT':
['DEV_SPLIT_x', 'DEV_SPLIT_y']

Development split columns:
['WELL', 'DEV_SPLIT', 'OFFICIAL_SPLIT']

Development split shape:
(98, 3)

Development split counts:
DEV_SPLIT
DEV_TRAIN         78
DEV_VALIDATION    20
Name: count, dtype: int64


In [16]:
# ============================================================
# V3 — CLEAN REBUILD OF DEVELOPMENT DATASETS
# ============================================================

print("=" * 70)
print("V3 — CLEAN DEVELOPMENT DATASET REBUILD")
print("=" * 70)

# ------------------------------------------------------------
# 1. Start from the frozen V3 ML table
# ------------------------------------------------------------

df_v3_clean = df_v3_ml.copy()

# Remove any development-split column created by previous cells.
# We will add exactly one fresh DEV_SPLIT column below.

split_columns_to_remove = [
    col for col in df_v3_clean.columns
    if col in ["DEV_SPLIT", "DEV_SPLIT_x", "DEV_SPLIT_y"]
]

if split_columns_to_remove:
    print(
        "\nRemoving old development columns:",
        split_columns_to_remove
    )

    df_v3_clean = df_v3_clean.drop(
        columns=split_columns_to_remove
    )

assert "DEV_SPLIT" not in df_v3_clean.columns

print("✓ Clean V3 dataframe prepared")

# ------------------------------------------------------------
# 2. Official FORCE split
# ------------------------------------------------------------

official_split = pd.read_csv(
    "../data/splits/force2020_split.csv"
)

official_train_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "TRAIN",
        "WELL"
    ]
)

public_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "PUBLIC",
        "WELL"
    ]
)

blind_test_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "BLIND_TEST",
        "WELL"
    ]
)

assert len(official_train_wells) == 98
assert len(public_wells) == 10
assert len(blind_test_wells) == 10

print("\nOfficial split:")
print(f"  TRAIN      : {len(official_train_wells)}")
print(f"  PUBLIC     : {len(public_wells)}")
print(f"  BLIND_TEST : {len(blind_test_wells)}")

# ------------------------------------------------------------
# 3. Keep only the 98 official TRAIN wells
# ------------------------------------------------------------

df_v3_train_pool = df_v3_clean[
    df_v3_clean["WELL"].isin(official_train_wells)
].copy()

assert df_v3_train_pool["WELL"].nunique() == 98

print("\n✓ Official TRAIN pool:")
print(f"  rows  = {len(df_v3_train_pool):,}")
print(f"  wells = {df_v3_train_pool['WELL'].nunique()}")

# ------------------------------------------------------------
# 4. Merge the already-frozen 78/20 development split
# ------------------------------------------------------------

df_v3_train_pool = df_v3_train_pool.merge(
    dev_split[["WELL", "DEV_SPLIT"]],
    on="WELL",
    how="left",
    validate="many_to_one",
)

assert "DEV_SPLIT" in df_v3_train_pool.columns
assert df_v3_train_pool["DEV_SPLIT"].notna().all()

print("✓ All 98 TRAIN wells mapped to DEV_SPLIT")

# ------------------------------------------------------------
# 5. Create development train / validation datasets
# ------------------------------------------------------------

df_dev_train = df_v3_train_pool[
    df_v3_train_pool["DEV_SPLIT"] == "DEV_TRAIN"
].copy()

df_dev_val = df_v3_train_pool[
    df_v3_train_pool["DEV_SPLIT"] == "DEV_VALIDATION"
].copy()

# ------------------------------------------------------------
# 6. Verify well counts
# ------------------------------------------------------------

train_wells = set(df_dev_train["WELL"].unique())
val_wells = set(df_dev_val["WELL"].unique())

print("\nDevelopment datasets:")
print(f"  DEV_TRAIN:")
print(f"    rows  = {len(df_dev_train):,}")
print(f"    wells = {len(train_wells)}")

print(f"\n  DEV_VALIDATION:")
print(f"    rows  = {len(df_dev_val):,}")
print(f"    wells = {len(val_wells)}")

assert len(train_wells) == 78
assert len(val_wells) == 20
assert train_wells.isdisjoint(val_wells)

print("\n✓ 78 / 20 well split confirmed")
print("✓ No well overlap")

# ------------------------------------------------------------
# 7. Verify PUBLIC / BLIND_TEST exclusion
# ------------------------------------------------------------

assert train_wells.isdisjoint(public_wells)
assert val_wells.isdisjoint(public_wells)

assert train_wells.isdisjoint(blind_test_wells)
assert val_wells.isdisjoint(blind_test_wells)

print("✓ PUBLIC wells excluded")
print("✓ BLIND_TEST wells excluded")

# ------------------------------------------------------------
# 8. Build final X / y matrices
# ------------------------------------------------------------

X_dev_train = df_dev_train[V3_FEATURES]
y_dev_train = df_dev_train[TARGET_COL]

X_dev_val = df_dev_val[V3_FEATURES]
y_dev_val = df_dev_val[TARGET_COL]

print("\nFinal V3 ML matrices:")
print(f"  X_dev_train : {X_dev_train.shape}")
print(f"  y_dev_train : {y_dev_train.shape}")
print(f"  X_dev_val   : {X_dev_val.shape}")
print(f"  y_dev_val   : {y_dev_val.shape}")

# ------------------------------------------------------------
# 9. Final feature-count checks
# ------------------------------------------------------------

assert X_dev_train.shape[1] == 76
assert X_dev_val.shape[1] == 76

assert list(X_dev_train.columns) == V3_FEATURES
assert list(X_dev_val.columns) == V3_FEATURES

print("\n✓ Exactly 76 V3 ML features")
print("✓ Feature order identical for train and validation")

print("\n" + "=" * 70)
print("✓ V3 DEVELOPMENT DATASET SUCCESSFULLY FROZEN")
print("=" * 70)

V3 — CLEAN DEVELOPMENT DATASET REBUILD

Removing old development columns: ['DEV_SPLIT']
✓ Clean V3 dataframe prepared

Official split:
  TRAIN      : 98
  PUBLIC     : 10
  BLIND_TEST : 10

✓ Official TRAIN pool:
  rows  = 1,171,557
  wells = 98
✓ All 98 TRAIN wells mapped to DEV_SPLIT

Development datasets:
  DEV_TRAIN:
    rows  = 894,965
    wells = 78

  DEV_VALIDATION:
    rows  = 276,592
    wells = 20

✓ 78 / 20 well split confirmed
✓ No well overlap
✓ PUBLIC wells excluded
✓ BLIND_TEST wells excluded

Final V3 ML matrices:
  X_dev_train : (894965, 76)
  y_dev_train : (894965,)
  X_dev_val   : (276592, 76)
  y_dev_val   : (276592,)

✓ Exactly 76 V3 ML features
✓ Feature order identical for train and validation

✓ V3 DEVELOPMENT DATASET SUCCESSFULLY FROZEN


In [17]:
# ============================================================
# V3 — FREEZE TRAINING-ONLY SQRT-BALANCED SAMPLE WEIGHTS
# ============================================================

from collections import Counter
import numpy as np
import pandas as pd

print("=" * 70)
print("V3 — TRAINING-ONLY CLASS WEIGHTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Class counts from DEV_TRAIN ONLY
# ------------------------------------------------------------

class_counts = (
    y_dev_train
    .value_counts()
    .sort_index()
)

N = len(y_dev_train)
K = len(class_counts)

print("\nDEV_TRAIN:")
print(f"  Samples : {N:,}")
print(f"  Classes : {K}")

print("\nClass counts:")
print(class_counts)

# ------------------------------------------------------------
# 2. Sqrt-balanced class weights
#
# w_c = sqrt(N / (K * N_c))
# ------------------------------------------------------------

sqrt_class_weights = np.sqrt(
    N / (K * class_counts)
)

print("\n" + "=" * 70)
print("SQRT-BALANCED CLASS WEIGHTS")
print("=" * 70)

for cls, weight in sqrt_class_weights.items():
    print(
        f"{int(cls):6d} : "
        f"count={class_counts[cls]:8,d}  "
        f"weight={weight:.6f}"
    )

# ------------------------------------------------------------
# 3. Convert class weights to per-sample weights
# ------------------------------------------------------------

sample_weights_v3 = (
    y_dev_train
    .map(sqrt_class_weights)
    .to_numpy(dtype=np.float64)
)

# ------------------------------------------------------------
# 4. Verify every sample received a weight
# ------------------------------------------------------------

assert len(sample_weights_v3) == len(y_dev_train)
assert np.isfinite(sample_weights_v3).all()
assert (sample_weights_v3 > 0).all()

print("\n✓ Every DEV_TRAIN sample has a valid weight")

# ------------------------------------------------------------
# 5. Weight statistics
# ------------------------------------------------------------

print("\nSample-weight statistics:")
print(f"  Minimum : {sample_weights_v3.min():.6f}")
print(f"  Maximum : {sample_weights_v3.max():.6f}")
print(f"  Mean    : {sample_weights_v3.mean():.6f}")
print(f"  Median  : {np.median(sample_weights_v3):.6f}")

# ------------------------------------------------------------
# 6. Verify the mapping class → weight
# ------------------------------------------------------------

weight_check = pd.DataFrame({
    "target": y_dev_train.to_numpy(),
    "weight": sample_weights_v3,
})

unique_weight_check = (
    weight_check
    .groupby("target")["weight"]
    .agg(["min", "max", "mean"])
)

print("\nWeight consistency by class:")
display(unique_weight_check)

# Every class must have exactly one weight
assert np.allclose(
    unique_weight_check["min"],
    unique_weight_check["max"]
)

# ------------------------------------------------------------
# 7. Save the frozen weight table for reproducibility
# ------------------------------------------------------------

V3_CLASS_WEIGHTS = {
    int(cls): float(weight)
    for cls, weight in sqrt_class_weights.items()
}

print("\n" + "=" * 70)
print("✓ V3 TRAINING WEIGHTS FROZEN")
print("=" * 70)

print("\nV3_CLASS_WEIGHTS:")
print(V3_CLASS_WEIGHTS)

V3 — TRAINING-ONLY CLASS WEIGHTS

DEV_TRAIN:
  Samples : 894,965
  Classes : 12

Class counts:
FORCE_2020_LITHOFACIES_LITHOLOGY
30000.0    131392
65000.0    544884
65030.0    121266
70000.0     43889
70032.0      7938
74000.0      1193
80000.0     22389
86000.0       845
88000.0      4315
90000.0      3306
93000.0       141
99000.0     13407
Name: count, dtype: int64

SQRT-BALANCED CLASS WEIGHTS
 30000 : count= 131,392  weight=0.753404
 65000 : count= 544,884  weight=0.369965
 65030 : count= 121,266  weight=0.784229
 70000 : count=  43,889  weight=1.303571
 70032 : count=   7,938  weight=3.065186
 74000 : count=   1,193  weight=7.906644
 80000 : count=  22,389  weight=1.825135
 86000 : count=     845  weight=9.394724
 88000 : count=   4,315  weight=4.157402
 90000 : count=   3,306  weight=4.749643
 93000 : count=     141  weight=22.998677
 99000 : count=  13,407  weight=2.358558

✓ Every DEV_TRAIN sample has a valid weight

Sample-weight statistics:
  Minimum : 0.369965
  Maximum : 22.

,min,max,mean
target,,,
30000.0,0.753404,0.753404,0.753404
65000.0,0.369965,0.369965,0.369965
65030.0,0.784229,0.784229,0.784229
70000.0,1.303571,1.303571,1.303571
70032.0,3.065186,3.065186,3.065186
74000.0,7.906644,7.906644,7.906644
80000.0,1.825135,1.825135,1.825135
86000.0,9.394724,9.394724,9.394724
88000.0,4.157402,4.157402,4.157402



✓ V3 TRAINING WEIGHTS FROZEN

V3_CLASS_WEIGHTS:
{30000: 0.7534040335745902, 65000: 0.36996477403109324, 65030: 0.7842289610211811, 70000: 1.303570608327514, 70032: 3.065186156036827, 74000: 7.9066439253009, 80000: 1.8251352563921224, 86000: 9.39472448378518, 88000: 4.157401827766058, 90000: 4.749643112859765, 93000: 22.998676598785163, 99000: 2.358558203159877}


In [18]:
# ============================================================
# V3 — CONTROLLED EXTRATREES EXPERIMENT
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

print("=" * 70)
print("V3 — EXTRATREES WITH FINER ROLLING FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify input dimensions
# ------------------------------------------------------------

print("\nInput:")
print(f"  Training samples   : {len(X_dev_train):,}")
print(f"  Validation samples : {len(X_dev_val):,}")
print(f"  Features            : {X_dev_train.shape[1]}")

assert X_dev_train.shape[1] == 76
assert X_dev_val.shape[1] == 76

# ------------------------------------------------------------
# 2. Define the controlled W5 ExtraTrees configuration
# ------------------------------------------------------------

model_v3_et = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight=None,
    n_jobs=-1,
    random_state=42,
)

print("\nModel:")
print(model_v3_et)

# ------------------------------------------------------------
# 3. Train using ONLY DEV_TRAIN
# ------------------------------------------------------------

print("\nTraining...")

start_time = time.perf_counter()

model_v3_et.fit(
    X_dev_train,
    y_dev_train,
    sample_weight=sample_weights_v3,
)

train_time = time.perf_counter() - start_time

print(f"✓ Training completed in {train_time:.2f} seconds")

# ------------------------------------------------------------
# 4. Predict DEV_VALIDATION
# ------------------------------------------------------------

print("\nPredicting DEV_VALIDATION...")

start_time = time.perf_counter()

y_pred_v3_et = model_v3_et.predict(X_dev_val)

prediction_time = time.perf_counter() - start_time

print(f"✓ Prediction completed in {prediction_time:.2f} seconds")

# ------------------------------------------------------------
# 5. Overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_dev_val,
    y_pred_v3_et
)

balanced_accuracy = balanced_accuracy_score(
    y_dev_val,
    y_pred_v3_et
)

macro_f1 = f1_score(
    y_dev_val,
    y_pred_v3_et,
    average="macro",
    zero_division=0,
)

weighted_f1 = f1_score(
    y_dev_val,
    y_pred_v3_et,
    average="weighted",
    zero_division=0,
)

print("\n" + "=" * 70)
print("V3 EXTRATREES RESULTS")
print("=" * 70)

print(f"\nAccuracy           : {accuracy:.6f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")

# ------------------------------------------------------------
# 6. Per-class classification report
# ------------------------------------------------------------

class_labels = sorted(
    np.unique(
        np.concatenate([
            y_dev_val.to_numpy(),
            y_pred_v3_et
        ])
    )
)

report = classification_report(
    y_dev_val,
    y_pred_v3_et,
    labels=class_labels,
    zero_division=0,
    output_dict=True,
)

report_df = pd.DataFrame(report).T

print("\n" + "=" * 70)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 70)

display(report_df)

# ------------------------------------------------------------
# 7. Store experiment summary
# ------------------------------------------------------------

v3_et_results = {
    "experiment": "V3_ExtraTrees_finer_rolling",
    "features": 76,
    "n_estimators": 100,
    "max_depth": 20,
    "min_samples_leaf": 5,
    "accuracy": accuracy,
    "balanced_accuracy": balanced_accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "train_time_sec": train_time,
    "prediction_time_sec": prediction_time,
}

print("\n" + "=" * 70)
print("✓ V3 EXTRATREES EXPERIMENT COMPLETE")
print("=" * 70)

V3 — EXTRATREES WITH FINER ROLLING FEATURES

Input:
  Training samples   : 894,965
  Validation samples : 276,592
  Features            : 76

Model:
ExtraTreesClassifier(max_depth=20, min_samples_leaf=5, n_jobs=-1,
                     random_state=42)

Training...
✓ Training completed in 32.84 seconds

Predicting DEV_VALIDATION...
✓ Prediction completed in 1.55 seconds

V3 EXTRATREES RESULTS

Accuracy           : 0.721937
Balanced Accuracy  : 0.417810
Macro F1           : 0.359845
Weighted F1        : 0.705209

PER-CLASS CLASSIFICATION REPORT


,precision,recall,f1-score,support
30000.0,0.584740,0.663484,0.621628,38013.000000
65000.0,0.828724,0.879167,0.853201,176301.000000
65030.0,0.324745,0.280514,0.301013,29339.000000
70000.0,0.609033,0.624809,0.616820,12431.000000
70032.0,0.000000,0.000000,0.000000,2575.000000
74000.0,0.149425,0.026263,0.044674,495.000000
80000.0,0.609614,0.200420,0.301664,10947.000000
86000.0,0.138085,0.775000,0.234405,240.000000
88000.0,0.640625,0.042062,0.078941,3899.000000
90000.0,0.514793,0.846304,0.640177,514.000000



✓ V3 EXTRATREES EXPERIMENT COMPLETE


In [19]:
# ============================================================
# V3 — XGBOOST WITH FINER ROLLING FEATURES
# ============================================================

import time
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

print("=" * 70)
print("V3 — XGBOOST WITH FINER ROLLING FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify inputs
# ------------------------------------------------------------

print("\nInput:")
print(f"  Training samples   : {len(X_dev_train):,}")
print(f"  Validation samples : {len(X_dev_val):,}")
print(f"  Features           : {X_dev_train.shape[1]}")

assert X_dev_train.shape[1] == 76
assert X_dev_val.shape[1] == 76

# ------------------------------------------------------------
# 2. Encode lithology labels as 0–11
# ------------------------------------------------------------

classes = np.array(sorted(y_dev_train.unique()))

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

y_train_xgb = y_dev_train.map(class_to_idx).to_numpy()
y_val_xgb = y_dev_val.map(class_to_idx).to_numpy()

assert len(classes) == 12
assert set(np.unique(y_train_xgb)) == set(range(12))

print("\nClasses:")
print(classes)

# ------------------------------------------------------------
# 3. XGBoost configuration
# ------------------------------------------------------------

model_v3_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=12,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

print("\nModel:")
print(model_v3_xgb)

# ------------------------------------------------------------
# 4. Train
# ------------------------------------------------------------

print("\nTraining...")

start_time = time.perf_counter()

model_v3_xgb.fit(
    X_dev_train,
    y_train_xgb,
    sample_weight=sample_weights_v3,
)

train_time = time.perf_counter() - start_time

print(f"✓ Training completed in {train_time:.2f} seconds")

# ------------------------------------------------------------
# 5. Predict
# ------------------------------------------------------------

print("\nPredicting DEV_VALIDATION...")

start_time = time.perf_counter()

y_pred_idx = model_v3_xgb.predict(X_dev_val)

prediction_time = time.perf_counter() - start_time

# Convert XGBoost class indices back to lithology codes
y_pred_v3_xgb = classes[y_pred_idx.astype(int)]

print(f"✓ Prediction completed in {prediction_time:.2f} seconds")

# ------------------------------------------------------------
# 6. Overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_dev_val,
    y_pred_v3_xgb
)

balanced_accuracy = balanced_accuracy_score(
    y_dev_val,
    y_pred_v3_xgb
)

macro_f1 = f1_score(
    y_dev_val,
    y_pred_v3_xgb,
    average="macro",
    zero_division=0,
)

weighted_f1 = f1_score(
    y_dev_val,
    y_pred_v3_xgb,
    average="weighted",
    zero_division=0,
)

print("\n" + "=" * 70)
print("V3 XGBOOST RESULTS")
print("=" * 70)

print(f"\nAccuracy           : {accuracy:.6f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")

# ------------------------------------------------------------
# 7. Per-class report
# ------------------------------------------------------------

report = classification_report(
    y_dev_val,
    y_pred_v3_xgb,
    labels=classes,
    zero_division=0,
    output_dict=True,
)

report_df_v3_xgb = pd.DataFrame(report).T

print("\n" + "=" * 70)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 70)

display(report_df_v3_xgb)

# ------------------------------------------------------------
# 8. Store experiment summary
# ------------------------------------------------------------

v3_xgb_results = {
    "experiment": "V3_XGBoost_finer_rolling",
    "features": 76,
    "n_estimators": 300,
    "max_depth": 3,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "accuracy": accuracy,
    "balanced_accuracy": balanced_accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "train_time_sec": train_time,
    "prediction_time_sec": prediction_time,
}

print("\n" + "=" * 70)
print("✓ V3 XGBOOST EXPERIMENT COMPLETE")
print("=" * 70)

V3 — XGBOOST WITH FINER ROLLING FEATURES

Input:
  Training samples   : 894,965
  Validation samples : 276,592
  Features           : 76

Classes:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 88000. 90000.
 93000. 99000.]

Model:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1, num_class=12, ...)

Training.

/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



V3 XGBOOST RESULTS

Accuracy           : 0.710595
Balanced Accuracy  : 0.409976
Macro F1           : 0.339586
Weighted F1        : 0.706856

PER-CLASS CLASSIFICATION REPORT


,precision,recall,f1-score,support
30000.0,0.599339,0.710809,0.650332,38013.000000
65000.0,0.859711,0.841391,0.850452,176301.000000
65030.0,0.323872,0.274958,0.297417,29339.000000
70000.0,0.559461,0.644116,0.598811,12431.000000
70032.0,0.064009,0.064854,0.064429,2575.000000
74000.0,0.059968,0.074747,0.066547,495.000000
80000.0,0.326696,0.331232,0.328949,10947.000000
86000.0,0.585185,0.329167,0.421333,240.000000
88000.0,0.676471,0.005899,0.011696,3899.000000
90000.0,0.394564,0.819066,0.532574,514.000000



✓ V3 XGBOOST EXPERIMENT COMPLETE


| Model / Features | Accuracy | Balanced Acc. | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| V2 + gradients ExtraTrees | .7201 | .4009 | .3028 | .7010 |
| **V3 ExtraTrees** | **.7219** | **.4178** | **.3598** | **.7052** |
| V2 + gradients XGBoost | .7127 | .4099 | **.3699** | .7082 |
| V3 XGBoost | .7106 | .4100 | .3396 | .7069 |

In [20]:
# ============================================================
# V3 — LIGHTGBM WITH FINER ROLLING FEATURES
# ============================================================

import time
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

print("=" * 70)
print("V3 — LIGHTGBM WITH FINER ROLLING FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify inputs
# ------------------------------------------------------------

print("\nInput:")
print(f"  Training samples   : {len(X_dev_train):,}")
print(f"  Validation samples : {len(X_dev_val):,}")
print(f"  Features           : {X_dev_train.shape[1]}")

assert X_dev_train.shape[1] == 76
assert X_dev_val.shape[1] == 76

# ------------------------------------------------------------
# 2. Encode lithology labels as 0–11
# ------------------------------------------------------------

classes = np.array(sorted(y_dev_train.unique()))

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

y_train_lgbm = y_dev_train.map(class_to_idx).to_numpy()
y_val_lgbm = y_dev_val.map(class_to_idx).to_numpy()

assert len(classes) == 12
assert set(np.unique(y_train_lgbm)) == set(range(12))

print("\nClasses:")
print(classes)

# ------------------------------------------------------------
# 3. LightGBM configuration
# ------------------------------------------------------------

model_v3_lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=12,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    n_jobs=-1,
    random_state=42,
    verbosity=-1,
)

print("\nModel:")
print(model_v3_lgbm)

# ------------------------------------------------------------
# 4. Train
# ------------------------------------------------------------

print("\nTraining...")

start_time = time.perf_counter()

model_v3_lgbm.fit(
    X_dev_train,
    y_train_lgbm,
    sample_weight=sample_weights_v3,
)

train_time = time.perf_counter() - start_time

print(f"✓ Training completed in {train_time:.2f} seconds")

# ------------------------------------------------------------
# 5. Predict
# ------------------------------------------------------------

print("\nPredicting DEV_VALIDATION...")

start_time = time.perf_counter()

y_pred_idx = model_v3_lgbm.predict(X_dev_val)

prediction_time = time.perf_counter() - start_time

# Convert class indices back to lithology codes
y_pred_v3_lgbm = classes[
    np.asarray(y_pred_idx).astype(int)
]

print(f"✓ Prediction completed in {prediction_time:.2f} seconds")

# ------------------------------------------------------------
# 6. Overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_dev_val,
    y_pred_v3_lgbm
)

balanced_accuracy = balanced_accuracy_score(
    y_dev_val,
    y_pred_v3_lgbm
)

macro_f1 = f1_score(
    y_dev_val,
    y_pred_v3_lgbm,
    average="macro",
    zero_division=0,
)

weighted_f1 = f1_score(
    y_dev_val,
    y_pred_v3_lgbm,
    average="weighted",
    zero_division=0,
)

print("\n" + "=" * 70)
print("V3 LIGHTGBM RESULTS")
print("=" * 70)

print(f"\nAccuracy           : {accuracy:.6f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")

# ------------------------------------------------------------
# 7. Per-class report
# ------------------------------------------------------------

report = classification_report(
    y_dev_val,
    y_pred_v3_lgbm,
    labels=classes,
    zero_division=0,
    output_dict=True,
)

report_df_v3_lgbm = pd.DataFrame(report).T

print("\n" + "=" * 70)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 70)

display(report_df_v3_lgbm)

# ------------------------------------------------------------
# 8. Store experiment summary
# ------------------------------------------------------------

v3_lgbm_results = {
    "experiment": "V3_LightGBM_finer_rolling",
    "features": 76,
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": -1,
    "num_leaves": 31,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1,
    "accuracy": accuracy,
    "balanced_accuracy": balanced_accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "train_time_sec": train_time,
    "prediction_time_sec": prediction_time,
}

print("\n" + "=" * 70)
print("✓ V3 LIGHTGBM EXPERIMENT COMPLETE")
print("=" * 70)

V3 — LIGHTGBM WITH FINER ROLLING FEATURES

Input:
  Training samples   : 894,965
  Validation samples : 276,592
  Features           : 76

Classes:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 88000. 90000.
 93000. 99000.]

Model:
LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
               n_jobs=-1, num_class=12, objective='multiclass', random_state=42,
               reg_lambda=1, subsample=0.8, verbosity=-1)

Training...
✓ Training completed in 247.93 seconds

Predicting DEV_VALIDATION...
✓ Prediction completed in 29.12 seconds

V3 LIGHTGBM RESULTS

Accuracy           : 0.713141
Balanced Accuracy  : 0.407197
Macro F1           : 0.347298
Weighted F1        : 0.710178


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



PER-CLASS CLASSIFICATION REPORT


,precision,recall,f1-score,support
30000.0,0.594488,0.725199,0.653370,38013.000000
65000.0,0.866949,0.829286,0.847699,176301.000000
65030.0,0.342220,0.348274,0.345220,29339.000000
70000.0,0.520130,0.693186,0.594317,12431.000000
70032.0,0.034211,0.010097,0.015592,2575.000000
74000.0,0.152047,0.052525,0.078078,495.000000
80000.0,0.370139,0.302549,0.332948,10947.000000
86000.0,0.616667,0.308333,0.411111,240.000000
88000.0,1.000000,0.000769,0.001538,3899.000000
90000.0,0.532258,0.770428,0.629571,514.000000



✓ V3 LIGHTGBM EXPERIMENT COMPLETE


| Metric | V2 + Gradients | V3 Finer Rolling | Change |
|---|---:|---:|---:|
| Accuracy | .713495 | .713141 | −.000354 |
| Balanced Acc. | .404474 | **.407197** | **+.002723** |
| Macro F1 | **.347732** | .347298 | −.000434 |
| Weighted F1 | .709796 | **.710178** | +.000382 |
| **65030 F1** | .341241 | **.345220** | **+.003979** |

| Model | Accuracy | Balanced Acc. | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| V2 + Grad ExtraTrees | .720122 | .400922 | .302777 | .700995 |
| **V3 ExtraTrees** | **.721937** | **.417810** | **.359845** | .705209 |
| V2 + Grad XGBoost | .712692 | .409914 | **.369926** | .708203 |
| V3 XGBoost | .710595 | .409976 | .339586 | .706856 |
| V2 + Grad LightGBM | .713495 | .404474 | .347732 | .709796 |
| V3 LightGBM | .713141 | .407197 | .347298 | **.710178** |

**now im going to do next experiment to imrove model that is...a class specific model**